<div style="padding: 15px; border: 1px solid #003152; border-radius: 5px; background-color: #fff5ef; color: #003152;">

**Project:** When Weight Doesn’t Weigh the same on Everyone: Applying The Lancet’s Clinical Obesity Framework to Map Phenotypic Diagnostic Gaps in NHANES 2021–2023  
**Source:** National Health and Nutrition Examination Survey (NHANES), Aug 2021–Aug 2023 Cycle  
**Key Transformation:** Vital Sign Aggregation, Adiposity Ratios, and Prognostic Risk Indexing  
**Version:** 1.0  

---
**Technical Focus:** Aggregation of baseline cardiovascular vitals to stabilize physiological variance due to . Calculates structural adiposity ratios alongside advanced cardiometabolic and hepatic risk equations to evaluate systemic impact.  
**Key Goal:** Categorize the cohort according to The Lancet's framework by transitioning from raw metrics to comprehensive diagnostic and cardiometabolic risk profiles.

</div>

In [ ]:
from IPython.display import display, Markdown, HTML
import os
import pyreadstat
import pandas as pd
import numpy as np
import pyarrow
from pathlib import Path

## 🧮 Parameter Aggregation & Clinical Framing
---

This notebook shifts focus from the initial database generation and curation to targeted clinical feature engineering. Within this phase, the `Processor` class is exclusively dedicated to aggregating baseline cardiovascular vitals, computing structural adiposity ratios, and operationalizing the clinical criteria for obesity as established by The Lancet Diabetes & Endocrinology Commission consensus framework.

In [91]:
class Loader:
    """
    Encapsulates file ingestion and cohort verification.
    """
    
    def __init__(self, data_dir: str = "../outputs"):
        self.data_dir = Path(data_dir)

    def base(self, filename: str = "nhanes_prepared_cohort.parquet") -> pd.DataFrame:
        path = self.data_dir / filename
        print(f"[DataLoader] Ingesting registry from: {path}")
        
        df = pd.read_parquet(path)
        
        # Immediate analytical check to guarantee no data degradation
        print(f"[DataLoader] Verification Successful. N = {df.shape[0]} rows.")
        return df

## ❤️ Cardiovascular Vital Signs Aggregation
---

To establish a stable and reliable baseline for cardiometabolic assessment, the Processor must aggregate multiple sequential readings of vital signs. Relying on a single measurement can introduce variance due to physiological fluctuations or environmental factors.

The class applies targeted aggregation logic to the following parameters:

- **Resting Pulse:** Evaluated and averaged to establish a baseline physiological rate.

- **Blood Pressure:** Both systolic and diastolic parameters are averaged across consecutive standardized readings. This mitigates transient spikes and isolates a resting cardiovascular baseline for downstream risk stratification.

## 📐 Derivation of adiposity ratios
---

Moving beyond absolute anthropometric measurements, the pipeline computes specific relative adiposity ratios. These derived variables provide a more nuanced representation of central fat distribution and metabolic risk compared to isolated metrics.

The pipeline computes the Waist-to-Height Ratio (WHtR) and the Waist-to-Hip Ratio (WHR) using the following mathematical formulations:

**Waist-to-Height Ratio (WHtR):** Captures relative visceral adiposity accumulation normalized to stature.

$$WHtR = \frac{W_c}{H}$$

**Waist-to-Hip Ratio (WHR):** Assesses gluteofemoral versus abdominal fat distribution mapping.

$$WHR = \frac{W_c}{H_c}$$

Where $W_c$ represents waist circumference, $H$ represents standing height, and $H_c$ represents hip circumference.

## 🗺️ Algorithmic Mapping of The Lancet Consensus Framework
---

To transition from basic anthropometric metrics to a comprehensive clinical phenotype, the Processor operationalizes the diagnostic pathways outlined by The Lancet Diabetes & Endocrinology Commission.

The pipeline evaluates a dual-pathway conditional matrix focusing strictly on the engineered anthropometric indices. Confirmation of excess or abnormal adiposity requires satisfying at least one of the two core strategic pathways defined by the Commission:

- **Synergistic Anthropometrics:** Presentation of at least one secondary anthropometric criterion—Waist Circumference ($W_c$), Waist-to-Hip Ratio ($WHR$), or Waist-to-Height Ratio ($WHtR$)—specifically in addition to an elevated baseline BMI ($\ge 30 \text{ kg/m}^2$, or $\ge 25 \text{ kg/m}^2$ where applicable).
- **BMI-Independent Adiposity:** Presentation of at least two distinct anthropometric criteria ($W_c$, $WHR$, or $WHtR$) met simultaneously, regardless of whether the patient's baseline BMI crosses traditional diagnostic thresholds.

To preserve computational simplicity and avoid over-fitting on arbitrary multi-layered demographic matrices, the Processor enforces standardized, universal clinical cut-offs across the cohort.


In [ ]:
class Processor:
    """
    Handles aggregation of multi-pass automated oscillometric measurements.
    """
    
    def __init__(self, dataframe):
        self.df = dataframe.copy()

        # Boolean mask for applying the scores
        self.mask = self.df["cohort"] == 1
        self.has_cohort = self.mask.any()

    def aggregation(self):
        """
        Compute robust row-wise averages.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        # Using pandas row-wise mean safely skips NaN elements vectorially
        self.df.loc[self.mask, 'sys_bp'] = df_cohort[['BPXOSY1', 'BPXOSY2', 'BPXOSY3']].mean(axis=1)
        self.df.loc[self.mask, 'dia_bp'] = df_cohort[['BPXODI1', 'BPXODI2', 'BPXODI3']].mean(axis=1)
        self.df.loc[self.mask, 'pulse_rate'] = df_cohort[['BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3']].mean(axis=1)
        
        return self
    
    def adiposity_ratios(self):
        """
        Derives clinical indexes mapping to visceral adiposity equations.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        # Waist-to-Hip Ratio (WHR)
        self.df.loc[self.mask, "WHR"] = df_cohort["BMXWAIST"] / df_cohort["BMXHIP"]
        # Waist-to-Height Ratio (WHtR)
        self.df.loc[self.mask, "WHtR"] = df_cohort["BMXWAIST"] / df_cohort["BMXHT"]
        
        return self
    
    def obesity_criteria(self):
        """
        Maps demographic-adjusted thresholds to isolate structural adiposity
        and defines explicit, mutually exclusive obesity phenotypes.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        # Ensure mandatory columns exist before evaluation
        required_cols = ["RIAGENDR", "BMXWAIST", "WHR", "WHtR", "BMXBMI"]

        if not all(col in df_cohort.columns for col in required_cols):
            raise KeyError("Missing essential anthropometric metrics required for classification.")
        
        # Demographics vector (1 = Male, 2 = Female)
        is_male = df_cohort["RIAGENDR"] == 1
        is_female = df_cohort["RIAGENDR"] == 2

        # Waist Circumference Thresholds (WHO Reference: Men > 102cm, Women > 88cm)
        high_wc = (is_male & (df_cohort["BMXWAIST"] > 102)) | (is_female & (df_cohort["BMXWAIST"] > 88))

        # Waist-to-Hip Ratio Thresholds (WHO Reference: Men > 0.90, Women > 0.85)
        high_whr = (is_male & (df_cohort["WHR"] > 0.90)) | (is_female & (df_cohort["WHR"] > 0.85))

        # Waist-to-Height Ratio Threshold (Universal Reference: >= 0.50)
        high_whtr = df_cohort["WHtR"] >= 0.50

        # Legacy Baseline (BMI >= 30)
        high_bmi = df_cohort["BMXBMI"] >= 30 
        normal_bmi_range = (df_cohort["BMXBMI"] >= 18.5) & (df_cohort["BMXBMI"] < 25)

        # Mathematically aggregate central adiposity markers
        df_cohort["anthro_criteria_count"] = sum([high_wc, high_whr, high_whtr])
        
        # Granular Phenotype Boolean Flags (Perfect for downstream Stratification/Regressions)
        # Phenotype A: Normal BMI but high central visceral fat deposition
        self.df.loc[self.mask, "phenotype_visceral"] = ((df_cohort["anthro_criteria_count"] >= 2) & (~high_bmi)).astype(bool)
        
        # Phenotype B: Elevated BMI paired with clinical central adiposity
        self.df.loc[self.mask, "phenotype_obesity"] = (high_bmi & (df_cohort["anthro_criteria_count"] >= 1)).astype(bool)
        
        # Phenotype C: High BMI but low central metrics (Athletes, Sarcopenic outliers)
        self.df.loc[self.mask, "phenotype_outliers"] = (high_bmi & (df_cohort["anthro_criteria_count"] == 0)).astype(bool)
        
        # Phenotype D: Clean, unconfounded baseline controls (Normal BMI & No central risk features)
        self.df.loc[self.mask, "phenotype_control"] = (normal_bmi_range & (df_cohort["anthro_criteria_count"] == 0)).astype(bool)

        return self

    def get_data(self):
        """
        Terminator method to break out of the chain and deliver the resulting DataFrame.
        """
        
        return self.df


### Kidney Function: Chronic Kidney Disease Epidemiology Collaboration (CKD-EPI 2021)

Establishes estimated Glomerular Filtration Rate (eGFR) without race-based coefficients to assess renal impairment and chronic kidney disease staging.

$$\text{eGFR} = \mu \times \min\left(\frac{S_{cr}}{\kappa}, 1\right)^{\alpha_1} \times \max\left(\frac{S_{cr}}{\kappa}, 1\right)^{\alpha_2} \times c^{\text{Age}} \times d$$

**Variables:**

* $\mu$: Intercept coefficient ($142$)
* $S_{cr}$: Serum Creatinine (`LBXSCR`) in $\text{mg/dL}$
* $\text{Age}$: Chronological age (`RIDAGEYR`) in years
* $c$: Age multiplier coefficient ($0.9938$)
* **Gender Parameter Matrix**:
    * **Female:** $\kappa = 0.7$, $\alpha_1 = -0.241$, $\alpha_2 = -1.200$, $d = 1.012$
    * **Male:** $\kappa = 0.9$, $\alpha_1 = -0.302$, $\alpha_2 = -1.200$, $d = 1.000$

*Reference: Inker LA, et al. New Creatinine- and Cystatin C-Based Equations to Estimate GFR without Race. N Engl J Med 2021.*

---

### Product of Fasting Glucose and Triglycerides (TyG Index)

A surrogate index for estimating insulin resistance.

$$\text{TyG} = \ln \left( \frac{\text{TG} \times \text{FBG}}{2} \right)$$

**Variables:**

* $\text{TG}$: Fasting Triglyceride concentration (`LBXTLG`) in $\text{mg/dL}$
* $\text{FBG}$: Fasting Blood Glucose concentration (`LBXGLU`) in $\text{mg/dL}$

**Clinical Cutoff:**

* A cutoff value of **$\ln(4.65) \approx 4.49$** is indicative of insulin resistance.

*Reference: Simental-Mendía LE, et al. The Product of Fasting Glucose and Triglycerides As Surrogate for Identifying Insulin Resistance. Sage Journals 2008.*

---

### Atherogenic Plasma Index (API)

Reflects the complex interactions of lipoprotein metabolism overall for predicting plasma atherogenicity.

$$\text{API} = \log_{10}\left(\frac{\text{TG}}{\text{HDL}_c}\right)$$

**Variables:**

* $\text{TG}$: Fasting Triglyceride concentration (`LBXTLG`) in $\text{mg/dL}$
* $\text{HDL}_c$: High-Density Lipoprotein Cholesterol concentration (`LBDHDD`) in $\text{mg/dL}$

**Clinical Cutoff:**

* An index value **$> 0.5$** indicates high atherogenic risk.

*Reference: Millán J, et al. Lipoprotein ratios. Vasc Health Risk Manag 2009.*

---

### Lipid Accumulation Product (LAP)

Describes anatomical and physiological lipid overaccumulation.

$$\text{LAP}_{\text{men}} = (\text{WC} - 65) \times (\text{TG} \times 0.01129)$$

$$\text{LAP}_{\text{women}} = (\text{WC} - 58) \times (\text{TG} \times 0.01129)$$

**Variables:**

* $\text{WC}$: Waist Circumference (`BMXWAIST`) in $\text{cm}$
* $\text{TG}$: Fasting Triglyceride concentration (`LBXTLG`) in $\text{mg/dL}$ *(multiplied by $0.01129$ to convert to $\text{mmol/L}$)*

**Clinical Cutoff:**

* Expressed along a population continuous scale; values exceeding the 75th percentile (typically **$> 47.3$ for men** and **$> 39.8$ for women**) strongly correlate with cardiovascular risk and metabolic syndrome.

*Reference: Kahn HS. The Lipid Accumulation Product. BMC Cardiovasc Disord 2005.*

---

### Metabolic Score for Insulin Resistance (METS-IR)

Indirect method for the detection of Insulin Resistance (IR), useful for prediction of incident Type 2 Diabetes.

$$\text{METS-IR} = \frac{\ln\left((2 \times G_0) + \text{TG}_0\right) \times \text{BMI}}{\ln(\text{HDL}_c)}$$

**Variables:**

* $G_0$: Fasting Glucose concentration (`LBXGLU`) in $\text{mg/dL}$
* $\text{TG}_0$: Fasting Triglyceride concentration (`LBXTLG`) in $\text{mg/dL}$
* $\text{HDL}_c$: High-Density Lipoprotein Cholesterol concentration (`LBDHDD`) in $\text{mg/dL}$
* $\text{BMI}$: Body Mass Index (`BMXBMI`) in $\text{kg/m}^2$

**Clinical Cutoff:**

* A score **$> 50.0$** serves as the validated diagnostic diagnostic threshold for insulin resistance.

*Reference: Bello-Chavolla OY, et al. METS-IR, a novel score to evaluate insulin sensitivity. BMC Endocr Disord 2018.*

---

### Fibrosis-4 Index (Fib-4)

Non-invasive screening metric for the prediction of hepatic fibrosis progression.

$$\text{Fib-4} = \frac{\text{Age} \times \text{AST}}{\text{Platelets} \times \sqrt{\text{ALT}}}$$

**Variables:**

* $\text{Age}$: Chronological age (`RIDAGEYR`) in years
* $\text{AST}$: Aspartate Aminotransferase (`LBXSAT`) in $\text{U/L}$
* $\text{ALT}$: Alanine Aminotransferase (`LBXSALT`) in $\text{U/L}$
* $\text{Platelets}$: Platelet count (`LBXPLT`) in $10^9/\text{L}$

**Clinical Cutoffs:**

* **$< 1.45$**: High negative predictive value to exclude advanced hepatic fibrosis.
* **$> 3.25$**: High positive predictive value indicating advanced hepatic fibrosis.

*Reference: Sterling RK, et al. Development of a Simple Noninvasive Index to Predict Significant Fibrosis. Hepatology 2006.*

---

### Fatty Liver Index (FLI)

Predictive algorithm determining the risk probability profile for hepatic steatosis.

$$\text{FLI} = \frac{100}{1 + e^{-y}}$$

$$y = 0.953 \ln(\text{TG}) + 0.139 \text{ BMI} + 0.718 \ln(\text{GGT}) + 0.053 \text{ WC} - 15.745$$

**Variables:**

* $\text{TG}$: Triglyceride concentration (`LBXTLG`) in $\text{mg/dL}$
* $\text{BMI}$: Body Mass Index (`BMXBMI`) in $\text{kg/m}^2$
* $\text{GGT}$: Gamma-Glutamyl Transferase (`LBXSGG`) in $\text{U/L}$
* $\text{WC}$: Waist Circumference (`BMXWAIST`) in $\text{cm}$

**Clinical Cutoffs:**

* **$< 30$**: Rules out hepatic steatosis (negative predictive value of 91%).
* **$\ge 60$**: Rules in hepatic steatosis (positive predictive value of 81%).

*Reference: Bedogni G, et al. The Fatty Liver Index. BMC Gastroenterol 2006.*

---

### Fibrotic NASH Index (FNI)

Detects fibrotic nonalcoholic steatohepatitis (NASH) among individuals at high risk for NAFLD/MASLD.

$$\text{FNI} = \frac{1}{1 + e^{-y}}$$

$$y = -10.33 + 2.54 \ln(\text{AST}) + 3.86 \ln(\text{HbA1c}) - 1.66 \ln(\text{HDL}_c)$$

**Variables:**

* $\text{AST}$: Aspartate Aminotransferase (`LBXSAT`) in $\text{U/L}$
* $\text{HbA1c}$: Glycated Hemoglobin (`LBXGH`) in $\%$
* $\text{HDL}_c$: High-Density Lipoprotein Cholesterol (`LBDHDD`) in $\text{mg/dL}$

**Clinical Cutoff:**

* A score **$\ge 0.10$** isolates individuals with significant fibrotic NASH (stages $\ge$ F2).

*Reference: Tavaglione F, et al. Development and Validation of a Score for Fibrotic Nonalcoholic Steatohepatitis. Clinical Gastroenterology and Hepatology 2022.*

---

### Predicting Risk of CVD Events (PREVENT - 10-Year Base Models)

Calculates the absolute 10-year risk of developing total cardiovascular disease (CVD, including stroke, myocardial infarction, and heart failure) using sex-differentiated equations.

#### 1. Standardization & Centering Transformation Step

Before computing log-odds, raw parameters must be scaled and centered around the pooled SI reference constants from Table S12.A:

* $\text{Age}_{\text{scaled}} = \dfrac{\text{Age} - 55}{10}$
* $\text{Non-HDL}_{\text{mmol/L}} = (\text{TC} \times 0.02586) - (\text{HDL}_c \times 0.02586) - 3.5$
* $\text{HDL}_{\text{scaled}} = \dfrac{(\text{HDL}_c \times 0.02586) - 1.3}{0.3}$
* $\text{SBP}_{\text{min\_scaled}} = \dfrac{\min(\text{SBP}, 110) - 110}{20}$
* $\text{SBP}_{\text{max\_scaled}} = \dfrac{\max(\text{SBP}, 110) - 130}{20}$
* $\text{eGFR}_{\text{min\_scaled}} = \dfrac{\min(\text{eGFR}, 60) - 60}{-15}$
* $\text{eGFR}_{\text{max\_scaled}} = \dfrac{\max(\text{eGFR}, 60) - 90}{-15}$

#### 2. Risk Evaluation Formulas

$$\text{10-Year CVD Risk (\%)} = \frac{100}{1 + e^{-\text{Log-Odds}}}$$

##### Log-Odds Formula for Women:
$$
\begin{aligned}
\text{Log-Odds}_{\text{women}} = &-3.3077 + 0.7939(\text{Age}_{\text{scaled}}) + 0.0305(\text{Non-HDL}_{\text{mmol/L}}) - 0.1607(\text{HDL}_{\text{scaled}}) \\
&- 0.2394(\text{SBP}_{\text{min\_scaled}}) + 0.3601(\text{SBP}_{\text{max\_scaled}}) + 0.8668(\text{Diabetes}) + 0.5361(\text{Smoker}) \\
&+ 0.6046(\text{eGFR}_{\text{min\_scaled}}) + 0.0434(\text{eGFR}_{\text{max\_scaled}}) + 0.3152(\text{Anti-HTN}) - 0.1478(\text{Statin}) \\
&- 0.0664(\text{Anti-HTN} \times \text{SBP}_{\text{max\_scaled}}) + 0.1198(\text{Statin} \times \text{Non-HDL}_{\text{mmol/L}}) \\
&- 0.0820(\text{Age}_{\text{scaled}} \times \text{Non-HDL}_{\text{mmol/L}}) + 0.0307(\text{Age}_{\text{scaled}} \times \text{HDL}_{\text{scaled}}) \\
&- 0.0946(\text{Age}_{\text{scaled}} \times \text{SBP}_{\text{max\_scaled}}) - 0.2706(\text{Age}_{\text{scaled}} \times \text{Diabetes}) \\
&- 0.0787(\text{Age}_{\text{scaled}} \times \text{Smoker}) - 0.1638(\text{Age}_{\text{scaled}} \times \text{eGFR}_{\text{min\_scaled}})
\end{aligned}
$$

##### Log-Odds Formula for Men:
$$
\begin{aligned}
\text{Log-Odds}_{\text{men}} = &-3.0312 + 0.7689(\text{Age}_{\text{scaled}}) + 0.0736(\text{Non-HDL}_{\text{mmol/L}}) - 0.0954(\text{HDL}_{\text{scaled}}) \\
&- 0.4347(\text{SBP}_{\text{min\_scaled}}) + 0.3363(\text{SBP}_{\text{max\_scaled}}) + 0.7693(\text{Diabetes}) + 0.4387(\text{Smoker}) \\
&+ 0.5379(\text{eGFR}_{\text{min\_scaled}}) + 0.0165(\text{eGFR}_{\text{max\_scaled}}) + 0.2889(\text{Anti-HTN}) - 0.1337(\text{Statin}) \\
&- 0.0476(\text{Anti-HTN} \times \text{SBP}_{\text{max\_scaled}}) + 0.1503(\text{Statin} \times \text{Non-HDL}_{\text{mmol/L}}) \\
&- 0.0518(\text{Age}_{\text{scaled}} \times \text{Non-HDL}_{\text{mmol/L}}) + 0.0191(\text{Age}_{\text{scaled}} \times \text{HDL}_{\text{scaled}}) \\
&- 0.1049(\text{Age}_{\text{scaled}} \times \text{SBP}_{\text{max\_scaled}}) - 0.2252(\text{Age}_{\text{scaled}} \times \text{Diabetes}) \\
&- 0.0895(\text{Age}_{\text{scaled}} \times \text{Smoker}) - 0.1544(\text{Age}_{\text{scaled}} \times \text{eGFR}_{\text{min\_scaled}})
\end{aligned}
$$
**Variables:**

* $\text{Age}$: Chronological age (`RIDAGEYR`) in years
* $\text{TC}$: Total Cholesterol (`LBXTC`) in $\text{mg/dL}$
* $\text{HDL}_c$: High-Density Lipoprotein Cholesterol (`LBDHDD`) in $\text{mg/dL}$
* $\text{SBP}$: Systolic Blood Pressure (`sys_bp` aggregated endpoint) in $\text{mmHg}$
* $\text{eGFR}$: Estimated GFR calculated via CKD-EPI 2021 in $\text{mL/min/1.73m}^2$
* $\text{Diabetes}$: Binary indicator ($1.0$ if diagnosed or `DIQ010` == 1, else $0.0$)
* $\text{Smoker}$: Binary current cigarette smoking status ($1.0$ if `SMQ040` $\in [1, 2]$, else $0.0$)
* $\text{Anti-HTN}$: Use of anti-hypertensive blood pressure medications ($1.0$ if `BPQ150` == 1, else $0.0$)
* $\text{Statin}$: Active cholesterol/statin medication usage ($1.0$ if `BPQ101D` == 1, else $0.0$)

*Reference: Khan SS, et al. Development and Validation of the American Heart Association’s PREVENT Equations. Circulation. 2024.*

In [93]:
class  Scores:
    """
    Calculation engine for clinical, cardiorenal, and hepatic indices.
    """

    def __init__(self, dataframe):
        self.df = dataframe.copy()

        # Boolean mask for applying the scores
        self.mask = self.df["cohort"] == 1
        self.has_cohort = self.mask.any()

    def ckd_epi(self):
        """
        Reference: Inker LA, et al. New Creatinine- and Cystatin C-Based Equations to Estimate GFR without Race. NEJM 2021. 
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        # Define parameters conditionally using numpy arrays
        is_male = df_cohort["RIAGENDR"] == 1
        kappa = np.where(is_male, 0.9, 0.7)
        alpha = np.where(is_male, -0.302, -0.241)
        gender_multiplier = np.where(is_male, 1.0, 1.012)
        
        scr_k = df_cohort["LBXSCR"] / kappa
        
        self.df.loc[self.mask, 'ckd_epi'] = (142 * 
            (np.minimum(scr_k, 1) ** alpha) * 
            (np.maximum(scr_k, 1) ** -1.200) * 
            (0.9938 ** df_cohort["RIDAGEYR"]) * 
            gender_multiplier)

        return self
    
    def tyg(self):
        """
        Triglycerides and fasting Glucose
        Reference: Simental-Mendía LE, et al. The Product of Fasting Glucose and Triglycerides As Surrogate for Identifying Insulin Resistance. Sage Journals 2008.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]
        
        self.df.loc[self.mask, 'score_tyg'] = np.log((df_cohort["LBXTLG"] * df_cohort["LBXGLU"]) / 2.0)
        
        return self

    def api(self):
        """
        Atherogenic Plasma Index
        Reference: Millán J, et al. Lipoprotein ratios. Vasc Health Risk Manag 2009.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        self.df.loc[self.mask, 'api'] = np.log10((df_cohort["LBXTLG"] / df_cohort["LBDHDD"]))
        
        return self

    def lap(self):
        """
        Lipid Accumulation Product
        Reference: Kahn HS. The Lipid Accumulation Product BMC Cardiovasc Disord 2005.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        # Convert to mmol/L
        tg_mmol = df_cohort["LBXTLG"] * 0.01129
        is_male = df_cohort["RIAGENDR"] == 1
        
        lap = np.where(
            is_male,
            (df_cohort["BMXWAIST"] - 65.0) * tg_mmol,
            (df_cohort["BMXWAIST"] - 58.0) * tg_mmol
        )
        
        self.df.loc[self.mask, 'score_lap'] = np.clip(lap, a_min=0, a_max=None)
        
        return self

    def mets_ir(self):
        """
        Metabolic Score for Insulin Resistance
        Reference: Bello-Chavolla OY, et al. METS-IR, a novel score to evaluate insulin sensitivity. BMC Endocr Disord 2018.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        numerator = np.log((2.0 * df_cohort["LBXGLU"]) + df_cohort["LBXTLG"]) * df_cohort["BMXBMI"]
        denominator = np.log(df_cohort["LBDHDD"])
        
        self.df.loc[self.mask,'score_mets_ir'] = numerator / denominator
        
        return self

    def fib4(self):
        """
        Fibrosis 4
        Reference: Sterling RK, et al. Development of a Simple Noninvasive Index to Predict Significant Fibrosis. Hepatology 2006.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        self.df.loc[self.mask,'fib4'] = (df_cohort["RIDAGEYR"] * df_cohort["LBXSASSI"]) / (df_cohort["LBXPLTSI"] * np.sqrt(df_cohort["LBXSATSI"]))
        
        return self

    def fli(self):
        """
        Fatty Liver Index
        Reference: Bedogni G, et al. The Fatty Liver Index. BMC Gastroenterol 2006.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        X = (0.953 * np.log(df_cohort["LBXTLG"]) + 
             (0.139 * df_cohort["BMXBMI"]) + 
             (0.718 * np.log(df_cohort["LBXSGTSI"])) + 
             (0.053 * df_cohort["BMXWAIST"]) - 15.745)
        self.df.loc[self.mask, 'fli'] = (100.0 / (1.0 + np.exp(-X)))
        
        return self

    def fni(self):
        """
        Fibrotic NASH Index
        Reference: Tavaglione F, et al. Development and Validation of a Score for Fibrotic Nonalcoholic Steatohepatitis. Clinical Gastroenterology and Hepatology 2022.
        """
        if not self.has_cohort:
            return self
        
        # Isolate the cohort slice using the centralized mask
        df_cohort = self.df[self.mask]

        X = (-10.33 + 
             (2.54 * np.log(df_cohort["LBXSASSI"])) + 
             (3.86 * np.log(df_cohort["LBXGH"])) - 
             (1.66 * np.log(df_cohort["LBDHDD"])))
        
        self.df.loc[self.mask, 'fni'] = 1.0 / (1.0 + np.exp(-X))

        return self

    def prevent_total(self):
        """
        Predicting Risk of CVD EVENTs
        Reference: Khan SS, et al. Development and Validation of the American Heart Association’s PREVENT Equations. Circulation. 2024.
        Calculate the exact 10-year Total CVD risk using base model constants from Table S12.A & Appendix 4 (Table S24).
        """
        if not self.has_cohort:
            return self

        # Isolate the current target cohort slice to prevent row mismatching
        df_cohort = self.df[self.mask]

        # Extract biomarkers and metrics
        # Convert the raw NHANES mg/dL values to mmol/L so they line up with the 3.5 and 1.3 constants
        age = df_cohort["RIDAGEYR"]
        tc = df_cohort["LBXTC"] * 0.02586       # Total Cholesterol (from mg/dL to mmol/L)
        hdl = df_cohort["LBDHDD"]* 0.02586      # HDL Cholesterol (from mg/dL to mmol/L)
        sbp = df_cohort["sys_bp"]               # Systolic Blood Pressure (mmHg)
        egfr = df_cohort["ckd_epi"]             # Estimated GFR via CKD-EPI
    
        # Binary exposure vectors (explicitly mapped to float flags 1.0 / 0.0)
        diabetes = np.where(df_cohort.get("DIQ010", 0) == 1, 1.0, 0.0)
        smoker = np.where(np.isin(df_cohort.get("SMQ040", 0), [1, 2]), 1.0, 0.0)
        is_bp_treated = np.where(df_cohort.get("BPQ150", 0) == 1, 1.0, 0.0)
        is_statin_user = np.where(df_cohort.get("BPQ101D", 0) == 1, 1.0, 0.0)

        # Compute standardized scaling transformations and piece-wise splines
        age_scaled = (age - 55) / 10
        non_hdl = tc - hdl - 3.5
        hdl_scaled = (hdl - 1.3) / 0.3
        
        sbp_min_scaled = (np.minimum(sbp, 110) - 110) / 20
        sbp_max_scaled = (np.maximum(sbp, 110) - 130) / 20
        
        egfr_min_scaled = (np.minimum(egfr, 60) - 60) / -15
        egfr_max_scaled = (np.maximum(egfr, 60) - 90) / -15

        # Compute Log-Odds Equation for Women (Table S24 Parameters)
        log_odds_women = (
            -3.307728 +
            0.7939329 * age_scaled +
            0.0305239 * non_hdl -
            0.1606857 * hdl_scaled -
            0.2394003 * sbp_min_scaled +
            0.360078 * sbp_max_scaled +
            0.8667604 * diabetes +
            0.5360739 * smoker +
            0.6045917 * egfr_min_scaled +
            0.0433769 * egfr_max_scaled +
            0.3151672 * is_bp_treated -
            0.1477655 * is_statin_user -
            0.0663612 * is_bp_treated * sbp_max_scaled +
            0.1197879 * is_statin_user * non_hdl -
            0.0819715 * age_scaled * non_hdl +
            0.0306769 * age_scaled * hdl_scaled -
            0.0946348 * age_scaled * sbp_max_scaled -
            0.27057 * age_scaled * diabetes -
            0.078715 * age_scaled * smoker -
            0.1637806 * age_scaled * egfr_min_scaled
        )

        # Compute Log-Odds Equation for Men (Table S24 Parameters)
        log_odds_men = (
            -3.031168 +
            0.7688528 * age_scaled +
            0.0736174 * non_hdl -
            0.0954431 * hdl_scaled -
            0.4347345 * sbp_min_scaled +
            0.3362658 * sbp_max_scaled +
            0.7692857 * diabetes +
            0.4386871 * smoker +
            0.5378979 * egfr_min_scaled +
            0.0164827 * egfr_max_scaled +
            0.288879 * is_bp_treated -
            0.1337349 * is_statin_user -
            0.0475924 * is_bp_treated * sbp_max_scaled +
            0.150273 * is_statin_user * non_hdl -
            0.0517874 * age_scaled * non_hdl +
            0.0191169 * age_scaled * hdl_scaled -
            0.1049477 * age_scaled * sbp_max_scaled -
            0.2251948 * age_scaled * diabetes -
            0.0895067 * age_scaled * smoker -
            0.1543702 * age_scaled * egfr_min_scaled
        )

        # Numerically stable logistic probability conversion
        # Maps log-odds to 0-100% absolute risk scale
        risk_women = 100.0 / (1.0 + np.exp(-log_odds_women))
        risk_men = 100.0 / (1.0 + np.exp(-log_odds_men))

        # Apply sex-differentiated mapping exclusively to the cohort mask
        # NHANES gender variable mapping: 1 = Male, 2 = Female
        is_female = self.df["RIAGENDR"] == 2
        self.df.loc[self.mask, 'prevent_total'] = np.where(
            is_female[self.mask], 
            risk_women, 
            risk_men
        )

        return self
    
    def get_data(self):
        """
        Terminator method to break out of the chain and deliver the resulting DataFrame.
        """
        
        return self.df

In [94]:
loader = Loader(data_dir="../outputs")
df = loader.base()

[DataLoader] Ingesting registry from: ..\outputs\nhanes_prepared_cohort.parquet
[DataLoader] Verification Successful. N = 11933 rows.


In [95]:
df_vitals = (
    Processor(df)
    .aggregation()
    .adiposity_ratios()
    .obesity_criteria()
    .get_data()
)

In [96]:
cohort = (
    Scores(df_vitals)
    .ckd_epi()
    .tyg()
    .api()
    .lap()
    .mets_ir()
    .fib4()
    .fli()
    .fni()
    .prevent_total()
    .get_data()
)

In [97]:
# Prepare the PREVENT predictor column specifically for the regression matrix
cohort['prevent_10pct'] = cohort['prevent_total'] / 10.0

In [ ]:
# # Export the cohort while embedding the metadata
# cohort.to_parquet(
#     '../outputs/nhanes_processed_cohort.parquet', 
#     engine='pyarrow', 
#     compression='snappy'
# )

In [ ]:
# # Load the dataset back into memory
# loaded_df = pd.read_parquet('../outputs/nhanes_processed_cohort.parquet')

# # Extract and inspect your clinical label mapping
# saved_metadata = loaded_df.attrs.get('clinical_metadata')

# # Quick test to see it worked
# print(type(saved_metadata))  # Output: <class 'dict'>

<class 'dict'>
